In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

def sql_query(query):
    return pd.read_sql(query, engine)

Que proyectos cuestan mas?

In [2]:
sql_query("""
    SELECT pj.nombre_proyecto,
           v.nombre_vertical,
           ROUND(100.0 * AVG((ap.resultado = 'Apto')::int), 1) AS pct_aptos
    FROM alumno_proyecto ap
    JOIN proyectos pj  ON pj.id_proyecto = ap.id_proyecto
    JOIN verticales v  ON v.id_vertical  = pj.id_vertical
    GROUP BY pj.nombre_proyecto, v.nombre_vertical
    ORDER BY v.nombre_vertical, pct_aptos
""")

,nombre_proyecto,nombre_vertical,pct_aptos
0,EDA,Data Science,56.0
1,BBDD,Data Science,72.0
2,HLF,Data Science,76.0
3,Deployment,Data Science,88.0
4,ML,Data Science,88.0
5,WebDev,Full Stack,59.3
6,FrontEnd,Full Stack,63.0
7,Backend,Full Stack,63.0
8,React,Full Stack,70.4
9,FullStack,Full Stack,74.1


Como le fue a cada promocion?

In [3]:
sql_query("""
    SELECT p.fecha_comienzo,
           c.nombre_campus,
           v.nombre_vertical,
           COUNT(DISTINCT m.id_matricula) AS alumnos,
           ROUND(100.0 * AVG((ap.resultado = 'Apto')::int), 1) AS pct_aptos
    FROM alumno_proyecto ap
    JOIN matriculas m  ON m.id_matricula = ap.id_matricula
    JOIN promociones p ON p.id_promocion = m.id_promocion
    JOIN campus c      ON c.id_campus    = p.id_campus
    JOIN verticales v  ON v.id_vertical  = p.id_vertical
    GROUP BY p.fecha_comienzo, c.nombre_campus, v.nombre_vertical
    ORDER BY p.fecha_comienzo, c.nombre_campus, v.nombre_vertical
""")

,fecha_comienzo,nombre_campus,nombre_vertical,alumnos,pct_aptos
0,2023-09-18,Madrid,Data Science,15,81.3
1,2023-09-18,Madrid,Full Stack,14,68.6
2,2024-02-12,Madrid,Data Science,10,68.0
3,2024-02-12,Valencia,Full Stack,13,63.1


Quién aprobo todo?

In [4]:
sql_query("""
    SELECT a.nombre_alumno, v.nombre_vertical
    FROM alumno_proyecto ap
    JOIN matriculas m  ON m.id_matricula = ap.id_matricula
    JOIN alumnos a     ON a.id_alumno    = m.id_alumno
    JOIN promociones p ON p.id_promocion = m.id_promocion
    JOIN verticales v  ON v.id_vertical  = p.id_vertical
    GROUP BY a.nombre_alumno, v.nombre_vertical
    HAVING COUNT(*) FILTER (WHERE ap.resultado = 'Apto') = COUNT(*)
    ORDER BY v.nombre_vertical, a.nombre_alumno
""")

,nombre_alumno,nombre_vertical
0,Anita Heredia,Data Science
1,Daniela Falcó,Data Science
2,Eli Casas,Data Science
3,Haydée Figueroa,Data Science
4,Leoncio Tena,Data Science
5,Rosalina Llanos,Data Science
6,Ernesto Uriarte,Full Stack
7,Herberto Figueras,Full Stack
8,Loida Arellano,Full Stack
9,Ricarda Tovar,Full Stack
